# 两端口单路径、增强响应和模拟反转

## 简介

本示例演示了一种快速方法，如果您正在使用无开关的三接收器系统来测量一个**互易**且**对称**的器件，就可以采用这种方法。有关这种架构类型的误差校正的更多信息，请参阅[使用三个接收器进行校准](Calibration With Three Receivers.ipynb)。通常，在无开关的三接收器架构上对二端口网络进行完全误差校正，需要以两种方向测量每个DUT。但是，如果已知DUT是互易的（$S_{21}=S_{12}$）并且是对称的（$S_{11}=S_{22}$），那么以两种方向进行的测量将产生相同的响应，因此是不必要的。以下工作示例比较了使用完全误差校正和伪完全误差校正对10 dB衰减器的校正响应，该衰减器在WR-12频率范围内进行测量：1. 完全校正2. 伪完全校正（FakeFlip）3. 部分校正（EnhancedResponse）

In [ ]:
from IPython.display import Image

Image('three_receiver_cal/pics/macgyver.jpg', width='50%')

## 示例

这些测量是在一台 Agilent PNAX 上进行的，使用了 VDI WR-12 TXRX-RX 频率扩展模块。校准标准和待测设备的测量数据通过将原始散射参数数据保存为触点文件，从 VNA 下载。在下面的代码中，将根据对应的测量值和校准标准的理想响应值，创建一个 TwoPortOnePath 校准。测量得到的网络数据从磁盘读取，而对应的理想响应值则使用 scikit-rf 生成。有关如何使用 scikit-rf 进行离线校准的更多信息，请参阅[此处](../../tutorials/Calibration.ipynb)。

In [ ]:
import matplotlib.pyplot as plt

import skrf as rf

%matplotlib inline

from skrf.calibration import TwoPortOnePath
from skrf.constants import mil
from skrf.media import RectangularWaveguide
from skrf.network import two_port_reflect as tpr

rf.stylely()

raw = rf.io.general.read_all_networks('three_receiver_cal/data/')



# pull frequency information from measurements
frequency = raw['short'].frequency

# the media object
wg = RectangularWaveguide(frequency=frequency, a=120*mil, z0_override=50)

# list of 'ideal' responses of the calibration standards
ideals = [wg.short(nports=2),
          tpr(wg.delay_short( 90,'deg'), wg.match()),
          wg.match(nports=2),
          wg.thru()]

# corresponding measurements to the 'ideals'
measured = [raw['short'],
            raw['quarter wave delay short'],
            raw['load'],
            raw['thru']]

# the Calibration object
cal = TwoPortOnePath(measured = measured, ideals = ideals )

## 校正选项

使用上述创建的校准数据，我们将比较使用“完全”、“伪完全”和“部分”校准方法校正后的 WR-12 10dB 衰减器的响应。以下将对每种校准算法进行描述。

In [ ]:
Image('three_receiver_cal/pics/symmetric DUT.jpg', width='75%')

### 完全校正（两端口单路径）

对于这种架构，完整的校正方法被称为 *TwoPortOnePath*。在 `scikit-rf` 中，使用此校正算法需要对器件进行两次测量，分别测量**正向**和**反向**，并将两者作为 `tuple` 传递给 `apply_cal()` 函数。忽略连接器的不确定性，这种校正方法与完整的两端口 **SOLT** 校准方法相同。

### 伪全校正（FakeFlip）

如果假设被测设备（DUT）是**互易**的并且**对称**的，那么在两种方向上测量该设备将产生相同的结果。因此，反向测量的结果可以用正向测量的结果进行复制来代替。我们将这种技术称为*虚假翻转*。

<div class="alert ">**警告**:在使用这种“临时解决方案”技术之前，请确保您理解互易性和对称性的假设，否则可能会导致产生无意义的结果。</div>

### 部分校正（增强响应）

如果将单个测量值传递给 `apply_cal()` 函数，则校准将采用部分校正。这种类型的校正被称为“增强响应”。虽然“假反转”技术假设器件是互易且对称的，但“增强响应”算法*隐含地*假设器件的端口 2 具有完美的匹配。使用这两种算法中的任何一种产生的校正结果的准确性取决于假设的准确性。

## 比较

In [ ]:
dutf = raw['attenuator (forward)']
dutr = raw['attenuator (reverse)']


# note the correction algorithm is different depending on what is passed to
# apply_cal
corrected_full =     cal.apply_cal((dutf, dutr))
corrected_fakeflip = cal.apply_cal((dutf,dutf))
corrected_partial =  cal.apply_cal(dutf)



f, ax = plt.subplots(2,2, figsize=(8,8))

for m in [0,1]:
    for n in [0,1]:
        ax_ = ax[m,n]
        ax_.set_title('$S_{%i%i}$'%(m+1,n+1))
        corrected_full.plot_s_db(m,n, label='Full Correction',ax=ax_ )
        corrected_fakeflip.plot_s_db(m,n, label='Pseudo-full Correction', ax=ax_)
        if n==0:
            corrected_partial.plot_s_db(m,n, label='Partial Correction', ax=ax_)


plt.tight_layout()